# 02 — Clinical score audit

**Purpose.** Validate and describe the existing score columns in the processed admission-level dataset. This notebook does not retrain models and does not overwrite `config['feature_sets']`.

An existing score column is treated as a supplied score value. It is not recalculated here from component variables. Any formula-specific recalculation should be implemented as reusable package code and compared with the supplied column.


In [12]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
from pathlib import Path
import yaml
from icare_risk.data.audit import load_admission_file
from icare_risk.data.clinical_scores import score_summary, validate_score_columns

CONFIG_PATH = Path('../../src/icare_risk/config/icare/analysis.yaml')
with CONFIG_PATH.open() as f:
    config = yaml.safe_load(f)

dataset_path = config['outputs']['processed_dataset']
df = load_admission_file(dataset_path)
score_definitions = config['scores']
score_columns = {
    name: definition['source_column']
    for name, definition in score_definitions.items()
}

print(f"Processed dataset: {dataset_path}")
print(f"Admissions: {len(df):,}")
print('Configured scores:')
for name, column in score_columns.items():
    print(f"  {name}: source column = {column}")

Processed dataset: data/processed/admissions_with_split.parquet
Admissions: 10,000
Configured scores:
  charlson_score: source column = charlson_score
  kim_score: source column = kim_score
  tumbarello_score: source column = tumbarello_score
  jones_score: source column = jones_score
  gavaghan_score: source column = gavaghan_score
  holmgren_score: source column = holmgren_score


In [3]:
validation = validate_score_columns(
    df=df, score_columns=list(score_columns.values()),
)
display(validation)
missing_scores = validation.loc[~validation['exists'], 'score'].tolist()
if missing_scores:
    raise KeyError(f"Configured score columns are missing: {missing_scores}")

,score,exists,n_missing,n_unique,min,max
0,charlson_score,True,0,13,0.0,13.0
1,kim_score,True,0,4,0.0,6.0
2,tumbarello_score,True,0,5,0.0,6.0
3,jones_score,True,0,2,0.0,2.0
4,gavaghan_score,True,0,2,0.0,1.0
5,holmgren_score,True,0,1,0.0,0.0


In [4]:
summary = score_summary(
    df=df,
    score_columns=list(score_columns.values()),
)
display(summary)


# Check score distributions by temporal split when the split label exists.
if 'split' in df.columns:
    print('Score distributions by split')
    for split_name, split_df in df.groupby('split', dropna=False):
        print(f"\n--- {split_name} ---")
        display(score_summary(split_df, list(score_columns.values())))

print('No configuration values were overwritten. Feature sets remain those defined in analysis.yaml.')

,score,n,n_missing,mean,std,median,min,max
0,charlson_score,10000,0,2.2592,1.743995,2.0,0.0,13.0
1,kim_score,10000,0,0.1164,0.368598,0.0,0.0,6.0
2,tumbarello_score,10000,0,0.2226,0.639914,0.0,0.0,6.0
3,jones_score,10000,0,0.0784,0.388160,0.0,0.0,2.0
4,gavaghan_score,10000,0,0.0006,0.024489,0.0,0.0,1.0
5,holmgren_score,10000,0,0.0000,0.000000,0.0,0.0,0.0


Score distributions by split

--- test ---


,score,n,n_missing,mean,std,median,min,max
0,charlson_score,2483,0,2.600483,1.775434,2.0,0.0,13.0
1,kim_score,2483,0,0.123641,0.390790,0.0,0.0,6.0
2,tumbarello_score,2483,0,0.232783,0.657030,0.0,0.0,6.0
3,jones_score,2483,0,0.085381,0.404397,0.0,0.0,2.0
4,gavaghan_score,2483,0,0.002416,0.049108,0.0,0.0,1.0
5,holmgren_score,2483,0,0.000000,0.000000,0.0,0.0,0.0



--- train ---


,score,n,n_missing,mean,std,median,min,max
0,charlson_score,6054,0,2.086389,1.711267,2.0,0.0,10.0
1,kim_score,6054,0,0.113314,0.351595,0.0,0.0,5.0
2,tumbarello_score,6054,0,0.219689,0.632797,0.0,0.0,4.0
3,jones_score,6054,0,0.074661,0.379173,0.0,0.0,2.0
4,gavaghan_score,6054,0,0.000000,0.000000,0.0,0.0,0.0
5,holmgren_score,6054,0,0.000000,0.000000,0.0,0.0,0.0



--- validation ---


,score,n,n_missing,mean,std,median,min,max
0,charlson_score,1463,0,2.395079,1.728027,2.0,0.0,9.0
1,kim_score,1463,0,0.116883,0.397507,0.0,0.0,5.0
2,tumbarello_score,1463,0,0.217362,0.640023,0.0,0.0,4.0
3,jones_score,1463,0,0.082023,0.396770,0.0,0.0,2.0
4,gavaghan_score,1463,0,0.000000,0.000000,0.0,0.0,0.0
5,holmgren_score,1463,0,0.000000,0.000000,0.0,0.0,0.0


No configuration values were overwritten. Feature sets remain those defined in analysis.yaml.


In [5]:
# Keep a stable, explicit list for later model feature-set definitions.
config['feature_sets'] = {
    name: {'columns': [definition['source_column']]}
    for name, definition in score_definitions.items()
}
print(config['feature_sets'])

{'charlson_score': {'columns': ['charlson_score']}, 'kim_score': {'columns': ['kim_score']}, 'tumbarello_score': {'columns': ['tumbarello_score']}, 'jones_score': {'columns': ['jones_score']}, 'gavaghan_score': {'columns': ['gavaghan_score']}, 'holmgren_score': {'columns': ['holmgren_score']}}
